<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 3: Covid Yarisan Grafik

**VERİ BİLİMİ TEMELLERİ** · Modül 3 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta03/hafta03_covid_yarisan_grafik.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta03/hafta03_covid_yarisan_grafik.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 3: COVID-19 Yarışan Çubuk Grafiği (Bar Chart Race)

Bu defterde COVID-19 vaka verilerini kullanarak **animasyonlu yarışan çubuk grafiği** oluşturacağız.

## İçindekiler
1. Gerekli kütüphanelerin kurulumu
2. Örnek veri oluşturma
3. Veriyi hazırlama (pivot tablo)
4. Yarışan çubuk grafiği animasyonu
5. MP4/GIF olarak kaydetme

## 1. Gerekli Kütüphanelerin Kurulumu

### bar_chart_race kütüphanesini yükleyelim

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# bar_chart_race kütüphanesini yükleyelim
!pip install bar_chart_race

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `bar_chart_race` | Yarışan çubuk grafik animasyonları |
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |


In [ ]:
import pandas as pd
import numpy as np
import bar_chart_race as bcr
import matplotlib.pyplot as plt

%matplotlib inline

print("Kütüphaneler başarıyla yüklendi!")

## 2. Gerçek COVID-19 Verisini Yükleme

Johns Hopkins Üniversitesi'nin açık kaynak COVID-19 veri setini kullanacağız. Bu veri seti, tüm dünya ülkelerinin günlük kümülatif vaka sayılarını içerir.

**Kaynak:** [Johns Hopkins CSSE COVID-19 Dataset](https://github.com/CSSEGISandData/COVID-19)

In [ ]:
# Johns Hopkins COVID-19 Onaylı Vakalar (Gerçek Veri)
url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
df_raw = pd.read_csv(url)

print(f"Veri seti boyutu: {df_raw.shape}")
print(f"Ülke sayısı: {df_raw['Country/Region'].nunique()}")
df_raw.head(3)

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
# Ülkelere göre grupla (bazı ülkelerin Province/State ayrımı var)
df_grouped = df_raw.groupby('Country/Region').sum(numeric_only=True)
df_grouped = df_grouped.drop(columns=['Lat', 'Long'], errors='ignore')

# En çok vakası olan 12 ülkeyi seç
top_countries = df_grouped.iloc[:, -1].nlargest(12).index.tolist()
df_top = df_grouped.loc[top_countries].T
df_top.index = pd.to_datetime(df_top.index)

# Haftalık örnekleme (daha temiz animasyon için)
df = df_top.resample('W').last()

# Ülke isimlerini Türkçeleştirelim
isim_haritasi = {
    'US': 'ABD', 'Brazil': 'Brezilya', 'India': 'Hindistan',
    'Russia': 'Rusya', 'France': 'Fransa', 'United Kingdom': 'İngiltere',
    'Turkey': 'Türkiye', 'Italy': 'İtalya', 'Spain': 'İspanya',
    'Germany': 'Almanya', 'Colombia': 'Kolombiya', 'Mexico': 'Meksika',
    'Argentina': 'Arjantin', 'Iran': 'İran', 'Indonesia': 'Endonezya',
    'Poland': 'Polonya', 'South Africa': 'Güney Afrika', 'Peru': 'Peru'
}
df.columns = [isim_haritasi.get(c, c) for c in df.columns]

print(f"Seçilen ülkeler: {list(df.columns)}")
print(f"Zaman aralığı: {df.index[0].strftime('%Y-%m-%d')} → {df.index[-1].strftime('%Y-%m-%d')}")
print(f"Veri boyutu: {df.shape}")
df.tail()

## 3. Veriyi İnceleme ve Hazırlama

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
# Son durumu inceleyelim
print("Son hafta kümülatif vaka sayıları (en yüksekten en düşüğe):")
print("=" * 40)
son_hafta = df.iloc[-1].sort_values(ascending=False)
for ulke, vaka in son_hafta.items():
    print(f"{ulke:15s}: {vaka:>12,}")

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Zaman içindeki değişimi görselleştirelim (statik grafik)
plt.figure(figsize=(14, 8))
for ulke in df.columns:
    plt.plot(df.index, df[ulke], linewidth=2, label=ulke)

plt.title("COVID-19 Kümülatif Vaka Sayıları (2020)", fontsize=16, fontweight='bold')
plt.xlabel("Tarih", fontsize=12)
plt.ylabel("Kümülatif Vaka Sayısı", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Yarışan Çubuk Grafiği Animasyonu

`bar_chart_race` kütüphanesi, DataFrame'i doğrudan animasyona dönüştürür.

**Önemli:** Verinin formatı şu şekilde olmalıdır:
- **Satırlar (index):** Tarihler
- **Sütunlar:** Ülkeler/Kategoriler
- **Değerler:** Sayısal veriler

Bizim verimiz zaten bu formatta!

In [ ]:
# Veri formatını doğrulayalım
print("Index tipi:", type(df.index))
print("Sütunlar:", list(df.columns))
print("\nVeri tipi kontrol:")
print(df.dtypes)

### Notebook içinde animasyon (HTML5 video olarak gösterir)

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Notebook içinde animasyon (HTML5 video olarak gösterir)
# NOT: Bu hücrenin çalışması birkaç dakika sürebilir

bcr.bar_chart_race(
    df=df,
    filename=None,  # None = notebook içinde göster
    n_bars=10,
    steps_per_period=10,
    period_length=500,
    title='COVID-19 Kümülatif Vaka Sayısı Yarışı (2020)',
    bar_size=0.9,
    figsize=(12, 7),
    period_fmt='%Y-%m-%d',
    period_label={'x': 0.95, 'y': 0.15, 'ha': 'right', 'fontsize': 14},
    bar_label_font=12,
    tick_label_font=11,
    bar_kwargs={'alpha': 0.8, 'lw': 0},
    filter_column_colors=True
)

## 5. Animasyonu Dosyaya Kaydetme

### 5.1 MP4 Olarak Kaydetme

MP4 kaydetmek için `ffmpeg` gereklidir. Yüklü değilse:
```
!pip install ffmpeg-python
# veya sistem düzeyinde:
# !apt-get install ffmpeg  (Linux)
# !brew install ffmpeg  (macOS)
```

In [ ]:
# MP4 olarak kaydet
bcr.bar_chart_race(
    df=df,
    filename='covid19_yarisan_grafik.mp4',
    n_bars=10,
    steps_per_period=10,
    period_length=500,
    title='COVID-19 Kümülatif Vaka Sayısı Yarışı (2020)',
    bar_size=0.9,
    figsize=(12, 7),
    period_fmt='%Y-%m-%d',
    period_label={'x': 0.95, 'y': 0.15, 'ha': 'right', 'fontsize': 14},
    bar_label_font=12,
    tick_label_font=11,
    bar_kwargs={'alpha': 0.8, 'lw': 0},
    filter_column_colors=True
)

print("✔ Animasyon 'covid19_yarisan_grafik.mp4' olarak kaydedildi!")

### 5.2 GIF Olarak Kaydetme

### GIF olarak kaydet

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# GIF olarak kaydet
bcr.bar_chart_race(
    df=df,
    filename='covid19_yarisan_grafik.gif',
    n_bars=10,
    steps_per_period=5,  # GIF için daha az adım (dosya boyutu küçülür)
    period_length=500,
    title='COVID-19 Kümülatif Vaka Sayısı Yarışı (2020)',
    bar_size=0.9,
    figsize=(10, 6),
    period_fmt='%Y-%m-%d',
    period_label={'x': 0.95, 'y': 0.15, 'ha': 'right', 'fontsize': 14},
    bar_label_font=11,
    tick_label_font=10,
    bar_kwargs={'alpha': 0.8, 'lw': 0},
    filter_column_colors=True
)

print("✔ Animasyon 'covid19_yarisan_grafik.gif' olarak kaydedildi!")

## 6. Özelleştirme İpuçları

### Renk Paleti Değiştirme

In [ ]:
# Özel renkler ile yarışan çubuk grafiği
ozel_renkler = {
    'ABD': '#B22234',
    'Brezilya': '#009C3B',
    'Hindistan': '#FF9933',
    'Rusya': '#0039A6',
    'Türkiye': '#E30A17',
    'İngiltere': '#012169',
    'Fransa': '#002395',
    'İtalya': '#008C45',
    'İspanya': '#AA151B',
    'Almanya': '#000000',
    'Kolombiya': '#FCD116',
    'Meksika': '#006341',
}

# Sütun sırasına göre renkleri diziye çevir
renk_listesi = [ozel_renkler[ulke] for ulke in df.columns]

bcr.bar_chart_race(
    df=df,
    filename=None,
    n_bars=10,
    steps_per_period=10,
    period_length=500,
    title='COVID-19 Kümülatif Vaka Yarışı (Bayrak Renkleri)',
    bar_size=0.9,
    figsize=(12, 7),
    period_fmt='%Y-%m-%d',
    period_label={'x': 0.95, 'y': 0.15, 'ha': 'right', 'fontsize': 14},
    bar_label_font=12,
    tick_label_font=11,
    cmap=renk_listesi,
    filter_column_colors=True
)

## Özet

Bu defterde öğrendiklerimiz:

- **bar_chart_race** kütüphanesinin kurulumu ve kullanımı
- Pandas DataFrame'den örnek zaman serisi verisi oluşturma
- Pivot tablo formatında veri hazırlama
- Animasyonlu yarışan çubuk grafiği oluşturma
- MP4 ve GIF formatında kaydetme
- Özel renkler ile özelleştirme

### Önemli Parametreler

| Parametre | Açıklama |
|-----------|----------|
| `filename` | Kayıt yolu (None = notebook içinde göster) |
| `n_bars` | Gösterilecek çubuk sayısı |
| `steps_per_period` | Her periyot arası geçiş adımı (akıcılık) |
| `period_length` | Her periyodun süresi (ms) |
| `cmap` | Renk haritası veya renk listesi |
| `filter_column_colors` | Renklerin sabit kalmasını sağlar |

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>